# Sales Demo - Charts Dashboard
Various chart types using the `LOAN_DB.PUBLIC.SALES_DEMO` table.

In [ ]:
from snowflake.snowpark.context import get_active_session
import matplotlib.pyplot as plt
import pandas as pd

session = get_active_session()

In [ ]:
%%sql -r sales_data
SELECT * FROM LOAN_DB.PUBLIC.SALES_DEMO ORDER BY SALE_DATE, REGION

## 1. Line Chart — Monthly Revenue Trend

In [ ]:
df = sales_data.copy()
monthly = df.groupby('SALE_DATE')[['REVENUE','PROFIT']].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly['SALE_DATE'], monthly['REVENUE'], marker='o', label='Revenue', linewidth=2)
ax.plot(monthly['SALE_DATE'], monthly['PROFIT'], marker='s', label='Profit', linewidth=2)
ax.set_title('Monthly Revenue & Profit Trend', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Amount ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. Bar Chart — Revenue by Region

In [ ]:
region = df.groupby('REGION')[['REVENUE','PROFIT']].sum().reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(region))
width = 0.35
ax.bar([i - width/2 for i in x], region['REVENUE'], width, label='Revenue', color='steelblue')
ax.bar([i + width/2 for i in x], region['PROFIT'], width, label='Profit', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(region['REGION'])
ax.set_title('Revenue & Profit by Region', fontsize=14)
ax.set_ylabel('Amount ($)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Pie Chart — Revenue Share by Product Category

In [ ]:
category = df.groupby('PRODUCT_CATEGORY')['REVENUE'].sum()

fig, ax = plt.subplots(figsize=(7, 7))
colors = ['#4e79a7', '#f28e2b', '#59a14f']
ax.pie(category, labels=category.index, autopct='%1.1f%%', colors=colors, startangle=90, textprops={'fontsize': 12})
ax.set_title('Revenue Share by Product Category', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Stacked Bar Chart — Monthly Revenue by Category

In [ ]:
pivot = df.pivot_table(index='SALE_DATE', columns='PRODUCT_CATEGORY', values='REVENUE', aggfunc='sum').fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', stacked=True, ax=ax, color=['#4e79a7', '#f28e2b', '#59a14f'])
ax.set_title('Monthly Revenue by Product Category', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Revenue ($)')
ax.set_xticklabels([d.strftime('%b %Y') for d in pivot.index], rotation=45)
ax.legend(title='Category')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Scatter Plot — Units Sold vs Revenue

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_map = {'Electronics': '#4e79a7', 'Clothing': '#f28e2b', 'Furniture': '#59a14f'}
for cat, grp in df.groupby('PRODUCT_CATEGORY'):
    ax.scatter(grp['UNITS_SOLD'], grp['REVENUE'], label=cat, color=colors_map[cat], s=80, alpha=0.7, edgecolors='white')
ax.set_title('Units Sold vs Revenue', fontsize=14)
ax.set_xlabel('Units Sold')
ax.set_ylabel('Revenue ($)')
ax.legend(title='Category')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Horizontal Bar Chart — Top Regions by Units Sold

In [ ]:
region_units = df.groupby('REGION')['UNITS_SOLD'].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(region_units.index, region_units.values, color='teal', edgecolor='white')
ax.set_title('Total Units Sold by Region', fontsize=14)
ax.set_xlabel('Units Sold')
for i, v in enumerate(region_units.values):
    ax.text(v + 10, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()